# 🏋️ Step 2: ペルソナ別 PPO 学習 → ONNX 生成

```
persona_rewards.json (Step1 で生成)
  ↓
ペルソナ A〜E を順番に PPO 学習 (各 ~60M steps)
  ↓
persona_A.onnx 〜 persona_E.onnx を単一ファイルで生成
  ↓
マイドライブ / mesa_persona_onnx / に保存
  ↓
step3_persona_city_sim.html で読み込む
```

## 修正済み問題
| 問題 | 対策 |
|------|------|
| `ModuleNotFoundError: onnxscript` | `onnxscript` をインストールに追加 |
| `LayerNormalization` opset 変換失敗 | `opset_version=17` に変更 |
| training mode 警告 | `model.eval()` + `torch.no_grad()` |
| `.onnx.data` 外部ファイル問題 | `onnx.save(save_as_external_data=False)` |
| GPU 使用率が低い | N_ENVS=4096, ROLLOUT=128, MINIBATCH=16384 に最適化 |

**Runtime: T4 GPU を選択してください**

In [ ]:
# ============================================================
# セル 1: インストール
# ============================================================
!pip install torch onnx onnxscript onnxruntime numpy -q
print('✓ Install complete')

In [ ]:
# ============================================================
# セル 2: インポート & GPU 確認 & Google Drive マウント
# ============================================================
import torch, torch.nn as nn, onnx
import numpy as np
import json, random, math, time, os, subprocess
from IPython.display import clear_output
import matplotlib.pyplot as plt, matplotlib

matplotlib.rcParams.update({
    'figure.facecolor': '#0a0c10', 'axes.facecolor': '#12161e',
    'text.color': '#c8d8e8',       'axes.labelcolor': '#c8d8e8',
    'xtick.color': '#4a5a6a',      'ytick.color': '#4a5a6a',
})

assert torch.cuda.is_available(), '❌ GPU 未検出。ランタイム → T4 GPU に変更してください'
DEVICE = torch.device('cuda')
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

def get_gpu_stats():
    try:
        r = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=2)
        v = r.stdout.strip().split(', ')
        return float(v[0]), float(v[1])/1024, float(v[2])/1024
    except:
        return 0.0, 0.0, 15.0

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/mesa_persona'       # persona_rewards.json の場所
ONNX_DIR = '/content/drive/MyDrive/mesa_persona_onnx'  # ONNX 出力先
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)
print(f'\n入力: {SAVE_DIR}')
print(f'出力: {ONNX_DIR}')

In [ ]:
# ============================================================
# セル 3: 定数 & ハイパーパラメータ
# ============================================================
OTHER=0; ROAD=1; BUILDING=2; TREE=3
GRID = 30

# GPU 最適化済みパラメータ (T4 / 15GB VRAM)
N_ENVS    = 4096
ROLLOUT   = 128
MINIBATCH = 16384   # Tensor Core 効率が最大になるサイズ
B         = N_ENVS * ROLLOUT  # 524,288 サンプル/更新

MAX_STEPS = 400
EPOCHS    = 4
CLIP_EPS  = 0.2
GAE_LAM   = 0.95
GAMMA     = 0.99
ENT_COEF  = 0.03
VF_COEF   = 0.5
LR        = 2e-4
STEPS_PER_PERSONA = 60_000_000

MOVE_DIST = 0.25
ROT_RAD   = math.radians(20.0)
RAY_DEG   = [-60, -30, 0, 30, 60]
N_RAYS    = 5
RAY_MAX   = 6.0
N_STEPS   = 60
RAY_STEP  = RAY_MAX / N_STEPS

RAY_D_T  = torch.linspace(RAY_STEP, RAY_MAX, N_STEPS, device=DEVICE)
RAY_DA_T = torch.tensor([math.radians(a) for a in RAY_DEG],
                         dtype=torch.float32, device=DEVICE)
OBS_DIM  = N_RAYS * 2 + 8  # 18
ACT_DIM  = 3               # 前進 / 左回転 / 右回転

print(f'バッチサイズ: {B:,}  OBS_DIM: {OBS_DIM}')
print(f'バッファ VRAM 見積もり: {ROLLOUT*N_ENVS*OBS_DIM*4*6/1e9:.2f}GB')

In [ ]:
# ============================================================
# セル 4: マップ生成
# ============================================================
def make_map(size=GRID, seed=42):
    rng  = random.Random(seed)
    grid = np.full((size, size), OTHER, dtype=np.int32)
    step = 4
    rows = list(range(0, size, step))
    cols = list(range(0, size, step))
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD
    for ri in range(len(rows)-1):
        for ci in range(len(cols)-1):
            r0,r1 = rows[ri]+1, rows[ri+1]
            c0,c1 = cols[ci]+1, cols[ci+1]
            cells = [(r,c) for r in range(r0,r1) for c in range(c0,c1)]
            if not cells: continue
            b = rng.choice(cells); grid[b[0],b[1]] = BUILDING
            for r,c in cells:
                if (r,c)==b: continue
                v = rng.random()
                if   v < 0.25: grid[r,c] = TREE
                elif v < 0.45: grid[r,c] = BUILDING
    for r in rows: grid[r,:] = ROAD
    for c in cols: grid[:,c] = ROAD
    return grid

MAP_NP    = make_map()
MAP_T     = torch.tensor(MAP_NP, dtype=torch.long,  device=DEVICE)
MAP_F     = MAP_T.float()
PASS_T    = (MAP_T == ROAD) | (MAP_T == BUILDING)
BCELLS_NP = np.array([(r,c) for r in range(GRID)
                             for c in range(GRID) if MAP_NP[r,c]==BUILDING])
BCELLS_T  = torch.tensor(BCELLS_NP, dtype=torch.float32, device=DEVICE)
NB        = len(BCELLS_NP)

cmap = matplotlib.colors.ListedColormap(['#1a3a6a','#d0d0c8','#c42020','#256b28'])
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(MAP_NP, cmap=cmap, vmin=0, vmax=3, origin='upper')
ax.set_title(f'City Map {GRID}x{GRID}  Buildings:{NB}', color='#c8d8e8')
ax.axis('off'); plt.tight_layout(); plt.show()
print(f'Buildings:{NB}  MAP_T:{MAP_T.device}')

In [ ]:
# ============================================================
# セル 5: GPU レイキャスト (TorchScript JIT)
# ============================================================
@torch.jit.script
def raycast_gpu(
    xs: torch.Tensor, ys: torch.Tensor, ths: torch.Tensor,
    ray_da: torch.Tensor, ray_d: torch.Tensor,
    map_t: torch.Tensor, grid: int, road: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    angles  = ths.unsqueeze(1) + ray_da.unsqueeze(0)
    dx = torch.cos(angles); dy = torch.sin(angles)
    px = xs[:,None,None] + dx[:,:,None] * ray_d[None,None,:]
    py = ys[:,None,None] + dy[:,:,None] * ray_d[None,None,:]
    in_b = (px>=0.0)&(px<float(grid))&(py>=0.0)&(py<float(grid))
    ri = px.long().clamp(0,grid-1); ci = py.long().clamp(0,grid-1)
    ct = map_t[ri,ci]; ct = torch.where(in_b, ct, torch.zeros_like(ct))
    is_hit  = ct != road
    first   = torch.argmax(is_hit.long(), dim=2)
    has_hit = is_hit.any(dim=2)
    first   = torch.where(has_hit, first, torch.full_like(first, ray_d.shape[0]-1))
    hit_ct  = ct.gather(2, first.unsqueeze(2)).squeeze(2).float()
    hit_d   = ray_d[first]
    hit_ct  = torch.where(has_hit, hit_ct, torch.ones_like(hit_ct)*float(road))
    return hit_ct/3.0, (hit_d/ray_d[-1]).clamp(0.0,1.0)

# warm-up
_x=torch.rand(N_ENVS,device=DEVICE)*GRID
_y=torch.rand(N_ENVS,device=DEVICE)*GRID
_t=torch.rand(N_ENVS,device=DEVICE)*math.pi*2
for _ in range(3): raycast_gpu(_x,_y,_t,RAY_DA_T,RAY_D_T,MAP_T,GRID,ROAD)
torch.cuda.synchronize()
t0=time.time()
for _ in range(50): raycast_gpu(_x,_y,_t,RAY_DA_T,RAY_D_T,MAP_T,GRID,ROAD)
torch.cuda.synchronize()
print(f'raycast JIT: {(time.time()-t0)/50*1000:.2f}ms/call ({N_ENVS} envs)')
del _x,_y,_t

In [ ]:
# ============================================================
# セル 6: ペルソナ対応ベクトル化環境
# CPU 同期ゼロ設計 (any()/item() を使わない)
# ============================================================
class PersonaVecEnv:
    def __init__(self, n, rp):
        self.n=n; self.rp=rp
        self.x    =torch.zeros(n,device=DEVICE); self.y    =torch.zeros(n,device=DEVICE)
        self.th   =torch.zeros(n,device=DEVICE)
        self.gx   =torch.zeros(n,device=DEVICE); self.gy   =torch.zeros(n,device=DEVICE)
        self.pdist=torch.zeros(n,device=DEVICE)
        self.stall=torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.scnt =torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.px   =torch.zeros(n,device=DEVICE); self.py   =torch.zeros(n,device=DEVICE)
        self.visited=torch.zeros(n,GRID,GRID,dtype=torch.uint8,device=DEVICE)
        self._rst(torch.ones(n,dtype=torch.bool,device=DEVICE))

    def _rand_b(self,mask):
        idx=torch.randint(0,NB,(self.n,),device=DEVICE)
        return BCELLS_T[idx,0]+0.5, BCELLS_T[idx,1]+0.5

    def _dist(self): return torch.sqrt((self.x-self.gx)**2+(self.y-self.gy)**2)

    def _rst(self,mask):
        bx,by=self._rand_b(mask); gx,gy=self._rand_b(mask)
        th=torch.rand(self.n,device=DEVICE)*math.pi*2
        nd=torch.sqrt((bx-gx)**2+(by-gy)**2)
        self.x    =torch.where(mask,bx,self.x);   self.y    =torch.where(mask,by,self.y)
        self.th   =torch.where(mask,th,self.th)
        self.gx   =torch.where(mask,gx,self.gx);  self.gy   =torch.where(mask,gy,self.gy)
        self.pdist=torch.where(mask,nd,self.pdist)
        self.stall=torch.where(mask,torch.zeros_like(self.stall),self.stall)
        self.scnt =torch.where(mask,torch.zeros_like(self.scnt), self.scnt)
        self.px   =torch.where(mask,bx,self.px);  self.py   =torch.where(mask,by,self.py)
        self.visited[mask]=0

    def reset_all(self):
        self._rst(torch.ones(self.n,dtype=torch.bool,device=DEVICE))
        return self._obs()

    def _obs(self):
        ht,hd=raycast_gpu(self.x,self.y,self.th,RAY_DA_T,RAY_D_T,MAP_T,GRID,ROAD)
        dx=self.gx-self.x; dy=self.gy-self.y
        ra=torch.atan2(dy,dx)-self.th
        dn=(torch.sqrt(dx**2+dy**2)/(GRID*math.sqrt(2))).clamp(0,1)
        ri=self.x.long().clamp(0,GRID-1); ci=self.y.long().clamp(0,GRID-1)
        cur=MAP_F[ri,ci]/3.0
        return torch.stack([
            ht[:,0],ht[:,1],ht[:,2],ht[:,3],ht[:,4],
            hd[:,0],hd[:,1],hd[:,2],hd[:,3],hd[:,4],
            torch.sin(ra),torch.cos(ra),dn,
            torch.sin(self.th),torch.cos(self.th),cur,
            (self.stall.float()/10).clamp(0,1),
            (self.scnt.float()/MAX_STEPS).clamp(0,1),
        ],dim=1).clamp(-1.0,1.0)

    def step(self,actions):
        rp=self.rp
        rew=torch.full((self.n,),-rp['step_penalty'],device=DEVICE)
        self.px=self.x.clone(); self.py=self.y.clone()

        # 回転
        rot=torch.where(actions==1,torch.full_like(self.th,-ROT_RAD),
            torch.where(actions==2,torch.full_like(self.th, ROT_RAD),
                        torch.zeros_like(self.th)))
        self.th=(self.th+rot)%(math.pi*2)

        # 前進
        fwd=(actions==0)
        nx=(self.x+torch.cos(self.th)*MOVE_DIST).clamp(0.01,GRID-0.01)
        ny=(self.y+torch.sin(self.th)*MOVE_DIST).clamp(0.01,GRID-0.01)
        ri=nx.long().clamp(0,GRID-1); ci=ny.long().clamp(0,GRID-1)
        passable=PASS_T[ri,ci]
        can_move=fwd&passable; blocked=fwd&~passable
        self.x=torch.where(can_move,nx,self.x); self.y=torch.where(can_move,ny,self.y)

        ri2=self.x.long().clamp(0,GRID-1); ci2=self.y.long().clamp(0,GRID-1)
        cur_cell=MAP_T[ri2,ci2]

        # 報酬
        rew=torch.where(can_move&(cur_cell==ROAD),  rew+rp['road_bonus'],     rew)
        rew=torch.where(cur_cell==BUILDING,          rew+rp['building_bonus'], rew)
        rew=torch.where(fwd,                         rew+rp['forward_bias'],   rew)
        rew=torch.where(blocked,                     rew-rp['violation_penalty'],rew)

        # 探索 / 再訪
        env_idx=torch.arange(self.n,device=DEVICE)
        was=self.visited[env_idx,ri2,ci2].bool()
        rew=torch.where(can_move&~was, rew+rp['explore_bonus'],   rew)
        rew=torch.where(can_move& was, rew-rp['revisit_penalty'], rew)
        self.visited[env_idx,ri2,ci2]=1

        # 社交ボーナス
        if rp['social_bonus']>0:
            ox=torch.roll(self.x,1); oy=torch.roll(self.y,1)
            near=(torch.sqrt((self.x-ox)**2+(self.y-oy)**2)<2.0)&can_move
            rew=torch.where(near,rew+rp['social_bonus'],rew)

        # 滞留
        moved=(torch.abs(self.x-self.px)+torch.abs(self.y-self.py))>0.05
        self.stall=torch.where(moved,torch.zeros_like(self.stall),self.stall+1)
        rew=torch.where(self.stall>3,rew-rp['stall_penalty'],rew)

        # 距離報酬
        nd=self._dist()
        rew=torch.where(nd<self.pdist,rew+0.5,rew)
        rew=torch.where(nd>self.pdist,rew-0.2,rew)
        self.pdist=nd

        # ゴール到達
        arrived=(cur_cell==BUILDING)&(nd<0.8)
        rew=torch.where(arrived,rew+rp['goal_reward'],rew)
        ngx,ngy=self._rand_b(arrived)
        self.gx=torch.where(arrived,ngx,self.gx); self.gy=torch.where(arrived,ngy,self.gy)
        self.pdist=torch.where(arrived,
            torch.sqrt((self.x-self.gx)**2+(self.y-self.gy)**2),self.pdist)

        self.scnt+=1
        done=self.scnt>=MAX_STEPS
        self._rst(done)
        return self._obs(),rew,done

print('PersonaVecEnv ✓')

In [ ]:
# ============================================================
# セル 7: ポリシーネットワーク
# ============================================================
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.ray_enc=nn.Sequential(
            nn.Linear(N_RAYS*2,128),nn.LayerNorm(128),nn.ReLU(),
            nn.Linear(128,128),nn.ReLU())
        self.nav_enc=nn.Sequential(nn.Linear(8,64),nn.ReLU())
        self.shared=nn.Sequential(
            nn.Linear(192,512),nn.LayerNorm(512),nn.ReLU(),
            nn.Linear(512,512),nn.LayerNorm(512),nn.ReLU(),
            nn.Linear(512,256),nn.ReLU())
        self.actor =nn.Linear(256,ACT_DIM)
        self.critic=nn.Linear(256,1)

    def forward(self,x):
        h=torch.cat([self.ray_enc(x[:,:N_RAYS*2]),
                     self.nav_enc(x[:,N_RAYS*2:])],dim=-1)
        h=self.shared(h); return self.actor(h),self.critic(h)

    def act(self,x):
        lg,val=self.forward(x)
        dist=torch.distributions.Categorical(torch.softmax(lg,-1))
        a=dist.sample(); return a,dist.log_prob(a),dist.entropy(),val.squeeze(-1)

n_params=sum(p.numel() for p in PolicyNet().parameters())
print(f'PolicyNet: {n_params:,} params')

In [ ]:
# ============================================================
# セル 8: ONNX 単一ファイルエクスポート関数
#   - opset_version=17 (LayerNorm 対応)
#   - eval() + no_grad() で training mode 警告なし
#   - save_as_external_data=False で .onnx.data を作らない
# ============================================================
def export_persona_onnx(policy, persona_id, rp):
    policy.eval()
    policy_cpu = policy.to('cpu')

    class ActorOnly(nn.Module):
        def __init__(self,p): super().__init__(); self.p=p
        def forward(self,x): l,_=self.p(x); return l

    actor=ActorOnly(policy_cpu); actor.eval()
    tmp_path  = f'/tmp/persona_{persona_id}_tmp.onnx'
    onnx_path = f'{ONNX_DIR}/persona_{persona_id}.onnx'

    # エクスポート
    with torch.no_grad():
        torch.onnx.export(
            actor, torch.zeros(1,OBS_DIM), tmp_path,
            input_names=['state'], output_names=['logits'],
            dynamic_axes={'state':{0:'batch'},'logits':{0:'batch'}},
            opset_version=17,
        )

    # 外部データを内部化して単一ファイルに統合
    onnx_model=onnx.load(tmp_path)
    onnx.save(onnx_model, onnx_path, save_as_external_data=False)

    # 後片付け
    for f in [tmp_path, tmp_path+'.data']:
        if os.path.exists(f): os.remove(f)

    size_mb=os.path.getsize(onnx_path)/1e6
    print(f'  ✓ persona_{persona_id}.onnx  ({size_mb:.1f}MB, 単一ファイル)')

    # 推論テスト
    import onnxruntime as ort
    sess=ort.InferenceSession(onnx_path,providers=['CPUExecutionProvider'])
    logits=sess.run(None,{'state':np.zeros((1,OBS_DIM),dtype=np.float32)})[0]
    print(f'  ✓ 推論テスト OK  logits={logits[0].round(3)}')

    # メタデータ保存
    meta={
        'persona_id':persona_id,
        'persona_name':rp.get('persona_name',f'Persona {persona_id}'),
        'description':rp.get('description',''),
        'reward_params':rp,
        'input_size':OBS_DIM,'output_size':ACT_DIM,
        'actions':['forward','turn_left','turn_right'],
        'move_dist':MOVE_DIST,'rot_deg':20.0,
        'ray_angles_deg':RAY_DEG,'ray_max_dist':RAY_MAX,
        'grid_size':GRID,
        'map':MAP_NP.tolist(),
        'building_cells':BCELLS_NP.tolist(),
    }
    meta_path=onnx_path.replace('.onnx','_meta.json')
    with open(meta_path,'w',encoding='utf-8') as f:
        json.dump(meta,f,indent=2,ensure_ascii=False)
    print(f'  ✓ persona_{persona_id}_meta.json 保存')

    policy.to(DEVICE)  # 次ペルソナ学習のため GPU に戻す
    return onnx_path

print('export_persona_onnx ✓')

In [ ]:
# ============================================================
# セル 9: 1ペルソナ分の学習関数
# ============================================================
def train_persona(persona_id, rp, total_steps=STEPS_PER_PERSONA):
    print('='*60)
    print(f'学習開始: [{persona_id}] {rp["persona_name"]}')
    print(f'  {rp["description"]}')
    print(f'  目標: {total_steps/1e6:.0f}M steps ({total_steps/B:.0f} updates)')
    print('='*60)

    policy   =PolicyNet().to(DEVICE)
    optimizer=torch.optim.Adam(policy.parameters(),lr=LR)
    env      =PersonaVecEnv(N_ENVS,rp)

    ckpt_path=f'{SAVE_DIR}/ckpt_{persona_id}.pt'
    start_update=0
    log={'reward':[],'trips':[],'viols':[],'explore':[],'gpu_util':[],'gpu_vram':[]}

    if os.path.exists(ckpt_path):
        ck=torch.load(ckpt_path,map_location=DEVICE)
        policy.load_state_dict(ck['policy'])
        optimizer.load_state_dict(ck['optimizer'])
        start_update=ck.get('update',0)
        log=ck.get('log',log)
        print(f'  ✓ 再開: update={start_update} ({start_update*B/1e6:.1f}M steps 済み)')

    obs=env.reset_all()

    # バッファ (GPU)
    obs_buf =torch.zeros(ROLLOUT,N_ENVS,OBS_DIM,device=DEVICE)
    act_buf =torch.zeros(ROLLOUT,N_ENVS,dtype=torch.long,device=DEVICE)
    rew_buf =torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)
    done_buf=torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)
    logp_buf=torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)
    val_buf =torch.zeros(ROLLOUT,N_ENVS,device=DEVICE)

    total=start_update*B; r_s=t_s=v_s=e_s=0.0; t0=time.time()
    LOG_EVERY=10

    for update in range(start_update, total_steps//B+1):

        # Rollout
        for t in range(ROLLOUT):
            with torch.no_grad():
                actions,logps,_,values=policy.act(obs)
            obs_buf[t]=obs; act_buf[t]=actions
            logp_buf[t]=logps; val_buf[t]=values
            obs,rew,done=env.step(actions)
            rew_buf[t]=rew; done_buf[t]=done.float()
            total+=N_ENVS
            r_s+=rew.sum().item()
            t_s+=(rew>=rp['goal_reward']*0.8).sum().item()
            v_s+=(rew<=-rp['violation_penalty']*0.8).sum().item()
            e_s+=(rew>=rp['explore_bonus']*0.8).sum().item()

        # GAE
        with torch.no_grad():
            _,lv=policy.forward(obs); lv=lv.squeeze(-1)
        adv=torch.zeros_like(rew_buf); gae=torch.zeros(N_ENVS,device=DEVICE)
        for t in reversed(range(ROLLOUT)):
            nv=lv if t==ROLLOUT-1 else val_buf[t+1]
            delta=rew_buf[t]+GAMMA*nv*(1-done_buf[t])-val_buf[t]
            gae=delta+GAMMA*GAE_LAM*(1-done_buf[t])*gae; adv[t]=gae
        ret=adv+val_buf

        # PPO
        obs_f=obs_buf.reshape(B,OBS_DIM); act_f=act_buf.reshape(B)
        logp_f=logp_buf.reshape(B); adv_f=adv.reshape(B); ret_f=ret.reshape(B)
        adv_f=(adv_f-adv_f.mean())/(adv_f.std()+1e-8)
        for _ in range(EPOCHS):
            perm=torch.randperm(B,device=DEVICE)
            for s in range(0,B,MINIBATCH):
                mb=perm[s:s+MINIBATCH]
                lg,vs=policy.forward(obs_f[mb])
                dist=torch.distributions.Categorical(torch.softmax(lg,-1))
                nlp=dist.log_prob(act_f[mb]); ent=dist.entropy().mean()
                ratio=torch.exp(nlp-logp_f[mb])
                lpi=-torch.min(ratio*adv_f[mb],
                    torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)*adv_f[mb]).mean()
                lvf=(vs.squeeze()-ret_f[mb]).pow(2).mean()
                loss=lpi+VF_COEF*lvf-ENT_COEF*ent
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(),0.5)
                optimizer.step()

        # ログ
        if update%LOG_EVERY==0 and update>0:
            d=LOG_EVERY*B
            ar=r_s/d; at=t_s/d*100; av=v_s/d*100; ae=e_s/d*100
            gpu_util,gpu_used,_=get_gpu_stats()
            sps=LOG_EVERY*B/(time.time()-t0)
            log['reward'].append(round(float(ar),4))
            log['trips'].append(round(float(at),3))
            log['viols'].append(round(float(av),3))
            log['explore'].append(round(float(ae),3))
            log['gpu_util'].append(round(float(gpu_util),1))
            log['gpu_vram'].append(round(float(gpu_used),3))
            r_s=t_s=v_s=e_s=0.0; t0=time.time()
            clear_output(wait=True)

            fig,axes=plt.subplots(1,5,figsize=(20,3))
            fig.patch.set_facecolor('#0a0c10')
            for ax,(k,c,title) in zip(axes,[
                ('reward','#00ddb4','Reward/step'),
                ('trips','#ffc840','GoalRate%'),
                ('viols','#ff5050','ViolRate%'),
                ('explore','#aa88ff','ExploreRate%'),
                ('gpu_util','#44aaff','GPU使用率%'),
            ]):
                v=log[k]
                if len(v)>=2: ax.plot(v,color=c,lw=1.5)
                ax.set_title(f'{title}\n{v[-1]:.2f}' if v else title,
                             color='#c8d8e8',fontsize=9)
                ax.grid(color='#1e2530',lw=0.5); ax.spines[:].set_color('#1e2530')
                if k=='gpu_util': ax.set_ylim(0,100)
            fig.suptitle(
                f'[{persona_id}] {rp["persona_name"]}  |  '
                f'Update:{update:,}  Steps:{total/1e6:.1f}M  '
                f'({total/total_steps*100:.1f}%)',
                color='#00ddb4',fontsize=11)
            plt.tight_layout(); plt.show()
            print(f'[{persona_id}] Upd:{update:5d} | R:{ar:7.4f} | '
                  f'Goal:{at:.2f}% | Viol:{av:.2f}% | Exp:{ae:.2f}% | '
                  f'{sps:.0f}sps | GPU:{gpu_util:.0f}% VRAM:{gpu_used:.2f}GB')

        # チェックポイント
        if update%50==0 and update>0:
            torch.save({'policy':policy.state_dict(),
                        'optimizer':optimizer.state_dict(),
                        'update':update,'log':log},ckpt_path)
            print(f'  💾 [{persona_id}] ckpt 保存 (update={update})')

    print(f'\n[{persona_id}] 学習完了 ({total/1e6:.1f}M steps)')
    return policy, log

print('train_persona ✓')

In [ ]:
# ============================================================
# セル 10: 報酬パラメータ読み込み
# ============================================================
rewards_path = f'{SAVE_DIR}/persona_rewards.json'
assert os.path.exists(rewards_path), (
    f'❌ {rewards_path} が見つかりません。\n'
    f'   step1_persona_reward_gen.ipynb を先に実行するか、\n'
    f'   persona_rewards.json (サンプル) を {SAVE_DIR} にアップロードしてください。'
)

with open(rewards_path, encoding='utf-8') as f:
    all_rewards = json.load(f)

# 数値を float に変換
FLOAT_KEYS = ['explore_bonus','building_bonus','road_bonus','forward_bias',
               'revisit_penalty','violation_penalty','goal_reward',
               'step_penalty','social_bonus','stall_penalty']
for pid,rp in all_rewards.items():
    for k in FLOAT_KEYS:
        rp[k] = float(rp.get(k, 0.0))

print(f'✓ {len(all_rewards)} ペルソナ読み込み完了')
print()
for pid,rp in all_rewards.items():
    print(f'  [{pid}] {rp["persona_name"]}')
    print(f'       explore={rp["explore_bonus"]:.1f}  '
          f'goal={rp["goal_reward"]:.1f}  '
          f'social={rp["social_bonus"]:.1f}  '
          f'step_pen={rp["step_penalty"]:.2f}')

In [ ]:
# ============================================================
# セル 11: 全ペルソナを順次学習 → ONNX エクスポート
# ============================================================
# 学習するペルソナを選択
# 全ペルソナ: list(all_rewards.keys())
# 途中から:   ['B','C','D','E']  など
TRAIN_PERSONAS = list(all_rewards.keys())

onnx_paths = {}

for pid in TRAIN_PERSONAS:
    rp = all_rewards[pid]

    # 学習
    policy, log = train_persona(pid, rp)

    # ONNX エクスポート
    print(f'\n[{pid}] ONNX エクスポート中...')
    path = export_persona_onnx(policy, pid, rp)
    onnx_paths[pid] = path

    # GPU メモリ解放
    del policy
    torch.cuda.empty_cache()
    print(f'\n→ [{pid}] 完了。次のペルソナへ...\n')
    time.sleep(2)

print('='*60)
print('✓ 全ペルソナの学習・エクスポート完了')
print(f'保存先: マイドライブ > mesa_persona_onnx')
print()
for pid, path in onnx_paths.items():
    size_mb = os.path.getsize(path)/1e6
    print(f'  [{pid}] persona_{pid}.onnx  ({size_mb:.1f}MB)')

In [ ]:
# ============================================================
# セル 12: 最終確認
# ============================================================
print('mesa_persona_onnx フォルダの内容:')
print()
total_mb = 0
for fname in sorted(os.listdir(ONNX_DIR)):
    fpath = f'{ONNX_DIR}/{fname}'
    mb    = os.path.getsize(fpath)/1e6
    total_mb += mb
    has_data = os.path.exists(fpath+'.data')
    flag = '⚠️  外部データあり (ブラウザ非対応)' if has_data else '✓ 単一ファイル'
    print(f'  {fname:<42} {mb:6.1f}MB  {flag}')

print(f'\n  合計: {total_mb:.1f}MB')
print()
print('次のステップ:')
print('  1. Google Drive から .onnx と _meta.json をダウンロード')
print('  2. step3_persona_city_sim.html をブラウザで開く')
print('  3. 各ペルソナの .onnx と _meta.json を同時に選択してロード')
print('  4. Start Simulation !')